# Tasca S10.01. APIs REST

## Integració de dades mitjançant APIs REST

Aquest quadern documenta la connexió a tres serveis web amb Python (`requests`) i el tractament de les respostes amb `pandas`. S'hi validen les operacions bàsiques (lectura, creació, modificació i esborrat), s'hi gestionen els codis d'estat i es documenta un flux d'extracció de dades fins a l'exportació en CSV.

**Estructura**
1. Validació del client HTTP sobre una API sandbox (JSONPlaceholder)
2. Ingesta de dades d'una API pública i estructuració en DataFrame
3. Extracció de dades obertes de l'Ajuntament de Barcelona i exportació a CSV

### Nivell 1: Validació del client HTTP sobre una API sandbox

**Objectiu:** comprovar que puc consultar recursos d'un servei web amb peticions GET i interpretar-ne les respostes.

**Servei:** [JSONPlaceholder](https://jsonplaceholder.typicode.com/), una API de proves.

In [10]:
# Importo les llibreries que necessito
import requests          # per fer peticions a una API
import pandas as pd      # per treballar amb taules (la faré servir més endavant)

import json              # per poder veure els camps del json de l'API que esculli

#### 1.1 Consulta de recursos: publicacions, usuaris i tasques

Consulto els tres recursos principals del servei mitjançant peticions GET.

In [2]:
# Guardo l'adreça base del servei en una variable
URL_BASE = "https://jsonplaceholder.typicode.com"

# Faig la petició GET al recurs "posts" i guardo la resposta
resposta_posts = requests.get(f"{URL_BASE}/posts")

# Faig el mateix amb "users"
resposta_users = requests.get(f"{URL_BASE}/users")

# Faig el mateix amb "todos"
resposta_todos = requests.get(f"{URL_BASE}/todos")

#### 1.2 Verificació del volum de dades i dels codis d'estat

Comprovo quants registres retorna cada recurs i si les peticions s'han resolt correctament.

In [3]:
# Converteixo la resposta de cada recurs en una llista de Python
dades_posts = resposta_posts.json()
dades_users = resposta_users.json()
dades_todos = resposta_todos.json()

# Mostro la quantitat de registres i el codi d'estat de cada petició
print("Posts:", len(dades_posts), "registres | codi d'estat:", resposta_posts.status_code)
print("Users:", len(dades_users), "registres | codi d'estat:", resposta_users.status_code)
print("Todos:", len(dades_todos), "registres | codi d'estat:", resposta_todos.status_code)

Posts: 100 registres | codi d'estat: 200
Users: 10 registres | codi d'estat: 200
Todos: 200 registres | codi d'estat: 200


#### 1.3 Gestió d'errors: consulta d'un recurs inexistent

Provoco una petició sobre una publicació que no existeix per comprovar com respon el servidor.

In [4]:
# Demano una publicació amb un identificador que no existeix
resposta_inexistent = requests.get(f"{URL_BASE}/posts/9999")

# Mostro el codi d'estat que retorna el servidor
print("Codi d'estat:", resposta_inexistent.status_code)

Codi d'estat: 404


#### 1.4 Creació d'un recurs (POST)

Envio una nova publicació al servei per validar el flux d'escriptura i reviso la resposta.

In [5]:
# Preparo les dades de la nova publicació en un diccionari
nova_publicacio = {
    "title": "Informe de proves d'integració",
    "body": "Publicació creada per validar el flux d'escriptura de l'API.",
    "userId": 1,
}

# Envio la petició POST amb les dades en format JSON
resposta_post = requests.post(f"{URL_BASE}/posts", json=nova_publicacio)

# Mostro la resposta en format JSON
print("Resposta JSON:", resposta_post.json())

# Mostro el codi d'estat
print("Codi d'estat:", resposta_post.status_code)

Resposta JSON: {'title': "Informe de proves d'integració", 'body': "Publicació creada per validar el flux d'escriptura de l'API.", 'userId': 1, 'id': 101}
Codi d'estat: 201


#### 1.5 Modificació parcial d'un recurs (PATCH)

Actualitzo només un camp d'una publicació existent i reviso la resposta del servei.

In [6]:
# Preparo només el camp que vull modificar
canvi_parcial = {"title": "Títol actualitzat"}

# Envio la petició PATCH sobre la publicació número 1
resposta_patch = requests.patch(f"{URL_BASE}/posts/1", json=canvi_parcial)

# Mostro la resposta en format JSON
print("Resposta JSON:", resposta_patch.json())

# Mostro el codi d'estat
print("Codi d'estat:", resposta_patch.status_code)

Resposta JSON: {'userId': 1, 'id': 1, 'title': 'Títol actualitzat', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}
Codi d'estat: 200


#### 1.6 Esborrat d'un recurs (DELETE)

Elimino una publicació existent i reviso la resposta del servei.

In [7]:
# Envio la petició DELETE sobre la publicació número 1
resposta_delete = requests.delete(f"{URL_BASE}/posts/1")

# Mostro la resposta en format JSON
print("Resposta JSON:", resposta_delete.json())

# Mostro el codi d'estat
print("Codi d'estat:", resposta_delete.status_code)

Resposta JSON: {}
Codi d'estat: 200


### Nivell 2: Ingesta de dades d'una API pública i estructuració en DataFrame

**Objectiu:** connectar amb un servei públic real, extreure'n dades en format JSON i estructurar-les en una taula.


#### 2.1 Selecció del servei

He seleccionat **PokéAPI** (https://pokeapi.co/) del repositori public-apis. És un servei públic, sense clau d'accés, amb HTTPS i respostes en JSON, i permet consultes GET.

#### 2.2 Revisió de la documentació

**Adreça base:** `https://pokeapi.co/api/v2/`

**Endpoints identificats:**
| Endpoint | Descripció |
|---|---|
| `/pokemon` | Llistat paginat de Pokémon (nom i enllaç al detall) |
| `/pokemon/{id o nom}` | Detall d'un Pokémon concret (pes, alçada, habilitats...) |
| `/type/{id o nom}` | Detall d'un tipus i els Pokémon que hi pertanyen |
| `/ability` | Llistat paginat d'habilitats |

**Filtres i paràmetres opcionals:**
| Paràmetre | Funció | Exemple |
|---|---|---|
| `limit` | Nombre de registres per pàgina (per defecte, 20) | `/pokemon?limit=60` |
| `offset` | Posició des d'on comença el llistat, per paginar | `/pokemon?limit=60&offset=60` |

**Format de resposta:** JSON (es verifica a la cel·la següent).

In [8]:
# Faig una petició GET a l'endpoint de llistat de Pokémon
resposta_api = requests.get("https://pokeapi.co/api/v2/pokemon")

# Mostro el tipus de contingut que indica el servidor a la capçalera
print("Tipus de contingut:", resposta_api.headers["Content-Type"])

Tipus de contingut: application/json; charset=utf-8


#### 2.3 Consulta a l'API i lectura de la resposta

Faig una petició GET a l'endpoint de Pokémon, en reviso el codi d'estat i n'extrec alguns camps.

In [11]:
# Defineixo l'adreça de l'endpoint i els paràmetres opcionals
url_pokemon = "https://pokeapi.co/api/v2/pokemon"
parametres = {"limit": 20, "offset": 0}

# Faig la petició GET passant els paràmetres
resposta_pokemon = requests.get(url_pokemon, params=parametres)

# Mostro el codi d'estat de la resposta
print("Codi d'estat:", resposta_pokemon.status_code)

# Converteixo la resposta a un diccionari de Python
contingut = resposta_pokemon.json()

# Imprimeixo alguns camps de la resposta de forma clara
print("Total de Pokémon disponibles:", contingut["count"])
print("Registres rebuts:", len(contingut["results"]))
print("Primer Pokémon:", contingut["results"][0]["name"])
print("Enllaç al detall:", contingut["results"][0]["url"])

Codi d'estat: 200
Total de Pokémon disponibles: 1351
Registres rebuts: 20
Primer Pokémon: bulbasaur
Enllaç al detall: https://pokeapi.co/api/v2/pokemon/1/


In [ ]:
# per veure els noms dels camps sense anar a l'API
print(json.dumps(contingut, indent=2))

{
  "count": 1351,
  "next": "https://pokeapi.co/api/v2/pokemon?offset=20&limit=20",
  "previous": null,
  "results": [
    {
      "name": "bulbasaur",
      "url": "https://pokeapi.co/api/v2/pokemon/1/"
    },
    {
      "name": "ivysaur",
      "url": "https://pokeapi.co/api/v2/pokemon/2/"
    },
    {
      "name": "venusaur",
      "url": "https://pokeapi.co/api/v2/pokemon/3/"
    },
    {
      "name": "charmander",
      "url": "https://pokeapi.co/api/v2/pokemon/4/"
    },
    {
      "name": "charmeleon",
      "url": "https://pokeapi.co/api/v2/pokemon/5/"
    },
    {
      "name": "charizard",
      "url": "https://pokeapi.co/api/v2/pokemon/6/"
    },
    {
      "name": "squirtle",
      "url": "https://pokeapi.co/api/v2/pokemon/7/"
    },
    {
      "name": "wartortle",
      "url": "https://pokeapi.co/api/v2/pokemon/8/"
    },
    {
      "name": "blastoise",
      "url": "https://pokeapi.co/api/v2/pokemon/9/"
    },
    {
      "name": "caterpie",
      "url": "https://

#### 2.4 Estructuració de les dades en un DataFrame

Converteixo els registres obtinguts en una taula amb pandas i en reviso les primeres files.

In [13]:
# Creo un DataFrame a partir de la llista de resultats
df_pokemon = pd.DataFrame(contingut["results"])

# Mostro les primeres files del DataFrame
df_pokemon.head()

,name,url
0,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/
1,ivysaur,https://pokeapi.co/api/v2/pokemon/2/
2,venusaur,https://pokeapi.co/api/v2/pokemon/3/
3,charmander,https://pokeapi.co/api/v2/pokemon/4/
4,charmeleon,https://pokeapi.co/api/v2/pokemon/5/


### Nivell 3: Extracció de dades obertes de l'Ajuntament de Barcelona

**Objectiu:** localitzar un conjunt de dades al catàleg d'Open Data BCN, extreure'n registres per API i exportar-los a CSV.

#### 3.1 Cerca d'un conjunt de dades al catàleg

Consulto el catàleg amb `package_search` i reviso el detall d'un conjunt amb `package_show`.

In [14]:
# Defineixo l'adreça base de l'API d'Open Data BCN
URL_BCN = "https://opendata-ajuntament.barcelona.cat/data/api/action"

# Cerco conjunts de dades que tinguin a veure amb l'arbrat (màxim 10)
resposta_cerca = requests.get(f"{URL_BCN}/package_search", params={"q": "arbrat", "rows": 10})

# Mostro el codi d'estat
print("Codi d'estat:", resposta_cerca.status_code)

# Guardo la llista de conjunts de dades trobats
conjunts = resposta_cerca.json()["result"]["results"]

# Recorro la llista i mostro el nom tècnic i el títol de cada conjunt
for conjunt in conjunts:
    print(conjunt["name"], "->", conjunt["title"])

Codi d'estat: 200
atles-biodiversitat-arbrat-rel-geometria-especie -> Geometry-species relationship of the trees of the Biodiversity Atlas of the city of Barcelona
atles-biodiversitat-arbrat-geometries -> Inventory of tree geometries of the Biodiversity Atlas of the city of Barcelona
atles-biodiversitat-arbrat-especies -> Inventory of tree species of the Atlas of Biodiversity of the city of Barcelona
arbrat-parcs -> Trees in the parks of the city of Barcelona
arbrat-zona -> Zone trees of the city of Barcelona
arbres-interes-local -> Local interest trees from Barcelona
arbrat-viari -> Street trees of the city of Barcelona


In [15]:
# Agafo el nom tècnic del primer conjunt de la llista
nom_conjunt = conjunts[0]["name"]

# Consulto les metadades d'aquest conjunt
resposta_detall = requests.get(f"{URL_BCN}/package_show", params={"id": nom_conjunt})

# Mostro el codi d'estat i el títol del conjunt
print("Codi d'estat:", resposta_detall.status_code)
print("Títol:", resposta_detall.json()["result"]["title"])

Codi d'estat: 200
Títol: Geometry-species relationship of the trees of the Biodiversity Atlas of the city of Barcelona


#### 3.2 Selecció d'un recurs amb dades descarregables

Reviso els recursos de cada conjunt trobat i identifico els que estan disponibles en CSV o JSON i consultables per API.

In [16]:
# Recorro tots els conjunts de dades trobats
for conjunt in conjunts:
    # Recorro els recursos (fitxers) de cada conjunt
    for recurs in conjunt["resources"]:
        # Em quedo només amb els recursos CSV o JSON
        if recurs.get("format", "").upper() in ["CSV", "JSON"]:
            # Mostro el conjunt, el format, si el datastore està actiu i l'identificador del recurs
            print(conjunt["name"], "|", recurs["format"], "| datastore actiu:", recurs.get("datastore_active"), "|", recurs["id"])

atles-biodiversitat-arbrat-rel-geometria-especie | CSV | datastore actiu: True | 506c3c5d-8f9b-409d-ae96-c0912da4a505
atles-biodiversitat-arbrat-rel-geometria-especie | JSON | datastore actiu: False | 77754ba7-b770-4c5b-95d8-67630bbe1c40
atles-biodiversitat-arbrat-geometries | CSV | datastore actiu: True | 7c64504a-a718-48fe-b466-8c93a160b0dd
atles-biodiversitat-arbrat-geometries | JSON | datastore actiu: False | b92299f8-47df-4045-8fc6-73a799d04db0
atles-biodiversitat-arbrat-especies | CSV | datastore actiu: True | c654c488-2822-4e51-ad9b-f443989a4b41
atles-biodiversitat-arbrat-especies | JSON | datastore actiu: False | a182e12d-039b-455a-bccb-ece021d4a98e
arbrat-parcs | JSON | datastore actiu: False | 3fde8f95-bd27-4f1d-85b6-16ca082b238e
arbrat-parcs | CSV | datastore actiu: True | 23076aaa-4f0e-4045-b4e5-61d5e651b5a6
arbrat-zona | JSON | datastore actiu: False | 7984c323-e2f5-47a0-af66-e03f03767494
arbrat-zona | CSV | datastore actiu: True | 29cd5c1f-11b1-404b-b3a5-ae29940b8c55
arbr

**Recurs seleccionat:** conjunt `arbrat-parcs` (arbres als parcs de la ciutat de Barcelona), recurs en format CSV amb el datastore actiu, consultable per API.

#### 3.3 Extracció de registres mitjançant l'API

Consulto el recurs seleccionat amb `datastore_search` i recupero 100 registres.

In [17]:
# Guardo l'identificador del recurs seleccionat (arbrat als parcs, CSV)
id_recurs = "23076aaa-4f0e-4045-b4e5-61d5e651b5a6"

# Demano 100 registres d'aquest recurs
resposta_dades = requests.get(
    f"{URL_BCN}/datastore_search",
    params={"resource_id": id_recurs, "limit": 100},
)

# Mostro el codi d'estat
print("Codi d'estat:", resposta_dades.status_code)

# Guardo la llista de registres
registres = resposta_dades.json()["result"]["records"]

# Mostro quants registres he obtingut i el primer
print("Registres obtinguts:", len(registres))
print("Primer registre:", registres[0])

Codi d'estat: 200
Registres obtinguts: 100
Primer registre: {'latitud': '41.4372530', 'cat_nom_catala': 'Pollancre gavatx', 'codi': '0000024AR', 'espai_verd': 'Central de Nou Barris, Parc', 'adreca': 'Pg Fabra i Puig, 450', 'catalogacio': '', 'nom_districte': 'NOU BARRIS', 'tipus_element': 'ARBRE PARC', 'cat_nom_castella': 'Chopo lombardo', 'nom_barri': 'LA GUINEUETA', 'geom': 'POINT (430266.320389286 4587634.30999226)', 'cat_nom_cientific': "Populus nigra 'Italica'", 'x_etrs89': '430266.320', 'data_plantacio': None, 'codi_barri': '48', 'longitud': '2.1653025', 'codi_districte': '8', 'tipus_reg': 'GOTEIG AVARIAT', 'categoria_arbrat': 'SEGONA', 'cat_especie_id': '152', 'tipus_aigua': '', 'y_etrs89': '4587634.310', '_id': 1}


#### 3.4 Estructuració de les dades en un DataFrame

Converteixo els registres obtinguts en una taula amb pandas i en reviso l'estructura.

In [18]:
# Creo un DataFrame a partir de la llista de registres
df_arbrat = pd.DataFrame(registres)

# Mostro les dimensions de la taula (files, columnes)
print("Dimensions:", df_arbrat.shape)

# Mostro les primeres files
df_arbrat.head()

Dimensions: (100, 23)


,latitud,cat_nom_catala,codi,espai_verd,adreca,catalogacio,nom_districte,tipus_element,cat_nom_castella,nom_barri,...,data_plantacio,codi_barri,longitud,codi_districte,tipus_reg,categoria_arbrat,cat_especie_id,tipus_aigua,y_etrs89,_id
0,41.4372530,Pollancre gavatx,0000024AR,"Central de Nou Barris, Parc","Pg Fabra i Puig, 450",,NOU BARRIS,ARBRE PARC,Chopo lombardo,LA GUINEUETA,...,NaN,48,2.1653025,8,GOTEIG AVARIAT,SEGONA,152,,4587634.310,1
1,41.4372866,-,0000025AR,"Central de Nou Barris, Parc","Pg Fabra i Puig, 450",,NOU BARRIS,ARBRE PARC,-,LA GUINEUETA,...,NaN,48,2.1653528,8,GOTEIG AVARIAT,PRIMERA,2336,,4587637.998,2
2,41.4373386,-,0000028AR,"Central de Nou Barris, Parc","Pg Fabra i Puig, 450",,NOU BARRIS,ARBRE PARC,-,LA GUINEUETA,...,NaN,48,2.1654466,8,GOTEIG AVARIAT,PRIMERA,2336,,4587643.703,3
3,41.4373646,Pollancre gavatx,0000029AR,"Central de Nou Barris, Parc","Pg Fabra i Puig, 450",,NOU BARRIS,ARBRE PARC,Chopo lombardo,LA GUINEUETA,...,NaN,48,2.1654937,8,GOTEIG AVARIAT,SEGONA,152,,4587646.550,4
4,41.4373825,Pollancre gavatx,0000030AR,"Central de Nou Barris, Parc","Pg Fabra i Puig, 450",,NOU BARRIS,ARBRE PARC,Chopo lombardo,LA GUINEUETA,...,NaN,48,2.1655264,8,GOTEIG AVARIAT,SEGONA,152,,4587648.513,5


#### 3.5 Exportació de les dades a CSV

Desa el DataFrame en un fitxer CSV perquè es pugui reutilitzar en altres anàlisis.

In [19]:
# Desa el DataFrame en un fitxer CSV, sense la columna d'índex i amb codificació UTF-8
df_arbrat.to_csv("arbrat_parcs_bcn.csv", index=False, encoding="utf-8")

# Confirmo que s'ha desat
print("Fitxer desat: arbrat_parcs_bcn.csv")

Fitxer desat: arbrat_parcs_bcn.csv


## Conclusions

- S'ha validat el cicle complet de peticions HTTP (GET, POST, PATCH i DELETE) sobre una API de proves, i s'han interpretat els codis d'estat (200, 201 i 404).
- S'han consultat dades d'una API pública (PokéAPI) i s'han estructurat en un DataFrame.
- S'ha localitzat un conjunt de dades d'Open Data BCN, s'han extret 100 registres amb `datastore_search` i s'han exportat a CSV.